**Pricing Simulation**

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [2]:
performance_df=pd.read_csv('performance_vehicle_insurance_dataset.csv')
inflation_df=pd.read_csv('inflation.csv')

In [3]:
performance_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 801838 entries, 0 to 801837
Data columns (total 20 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   SEX                801838 non-null  int64  
 1   INSR_BEGIN         801838 non-null  object 
 2   INSR_END           801838 non-null  object 
 3   EFFECTIVE_YR       801834 non-null  object 
 4   INSR_TYPE          801838 non-null  int64  
 5   INSURED_VALUE      801838 non-null  float64
 6   PREMIUM            801838 non-null  float64
 7   OBJECT_ID          801838 non-null  int64  
 8   PROD_YEAR          801669 non-null  float64
 9   SEATS_NUM          801603 non-null  float64
 10  CARRYING_CAPACITY  603676 non-null  float64
 11  TYPE_VEHICLE       801838 non-null  object 
 12  CCM_TON            801830 non-null  float64
 13  MAKE               801838 non-null  object 
 14  USAGE              801838 non-null  object 
 15  CLAIM_PAID         801838 non-null  float64
 16  PR

In [4]:
performance_df=performance_df[['INSR_BEGIN','INSURED_VALUE','PREMIUM','TYPE_VEHICLE','CLAIM_PAID','has_claim','begin_year','policy_duration']].copy()

In [5]:
performance_df[performance_df['policy_duration']==0]

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration
14593,2014-05-25,0.0,790.63,Automobile,0.0,0,2014,0
48016,2014-05-25,0.0,575.00,Automobile,0.0,0,2014,0
50370,2014-05-25,0.0,661.25,Automobile,0.0,0,2014,0
50371,2014-05-25,0.0,661.25,Automobile,0.0,0,2014,0
50373,2014-05-25,0.0,575.00,Automobile,0.0,0,2014,0
...,...,...,...,...,...,...,...,...
698607,2016-04-17,0.0,631.55,Automobile,0.0,0,2016,0
698608,2016-04-17,0.0,631.55,Automobile,0.0,0,2016,0
698609,2016-04-17,0.0,631.55,Automobile,0.0,0,2016,0
698610,2016-04-17,0.0,631.55,Automobile,0.0,0,2016,0


In [6]:
performance_df=performance_df[performance_df['policy_duration']!=0]

In [7]:
performance_df[performance_df['policy_duration']==0]

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration


In [8]:
inflation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   year                 8 non-null      int64  
 1   inflation_rate       8 non-null      float64
 2   factor_to_2017_base  8 non-null      float64
dtypes: float64(2), int64(1)
memory usage: 324.0 bytes


In [9]:
performance_df=performance_df.merge(inflation_df[['year','factor_to_2017_base']], left_on='begin_year', right_on='year', how='left' )

In [10]:
performance_df.head()

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration,year,factor_to_2017_base
0,2013-08-08,519755.22,7209.140,Pick-Up,0.00,0,2013,364,2013,1.382345
1,2012-08-08,519755.22,7203.890,Pick-Up,0.00,0,2012,364,2012,1.485468
2,2011-08-08,519755.22,7045.804,Pick-Up,0.00,0,2011,365,2011,1.836039
3,2011-07-08,519755.22,287.250,Pick-Up,0.00,0,2011,30,2011,1.836039
4,2013-08-08,285451.24,4286.900,Pick-Up,19894.43,1,2013,364,2013,1.382345


In [11]:
performance_df=performance_df.drop(columns=['year'])

In [12]:
performance_df['Claim_Adj']=performance_df['CLAIM_PAID']*performance_df['factor_to_2017_base']
performance_df['Premium_Adj']=performance_df['PREMIUM']*performance_df['factor_to_2017_base']
performance_df['Insured_Adj']=performance_df['INSURED_VALUE']*performance_df['factor_to_2017_base']

In [13]:
performance_df['exposure']=performance_df['policy_duration']/365

In [14]:
risk_by_year_vehicle_raw = (
    performance_df.groupby(["begin_year", "TYPE_VEHICLE"])
    .agg(
        sum_claims_adj=("Claim_Adj", "sum"),
        sum_premium_adj=("Premium_Adj", "sum"),
        sum_Policy_duration=("policy_duration", "sum"),
        count_claims=("has_claim", "sum")
    )
    .reset_index()
)
risk_by_year_vehicle_raw['exposure']=risk_by_year_vehicle_raw['sum_Policy_duration']/365


In [15]:
risk_prev=risk_by_year_vehicle_raw.copy()
risk_prev['frequency']=risk_prev['count_claims']/risk_prev['exposure']
risk_prev['severity'] = np.where(
    risk_prev['count_claims'] > 0,
    risk_prev['sum_claims_adj'] / risk_prev['count_claims'],
    0
)
risk_prev['expected_loss'] =risk_prev['frequency']* risk_prev['severity']

In [16]:
risk_prev

,begin_year,TYPE_VEHICLE,sum_claims_adj,sum_premium_adj,sum_Policy_duration,count_claims,exposure,frequency,severity,expected_loss
0,2011,Automobile,1.302647e+08,5.571959e+07,4769338,861,13066.679452,0.065893,151294.678552,9969.228886
1,2011,Bus,5.833444e+08,1.190773e+08,3189598,954,8738.624658,0.109170,611472.153401,66754.719101
2,2011,Motor-Cycle,1.183300e+06,8.457813e+06,1957333,73,5362.556164,0.013613,16209.583027,220.659612
3,2011,Pick-Up,2.102559e+08,7.897389e+07,3939460,1060,10793.041096,0.098211,198354.635264,19480.692375
4,2011,Special Construction,9.416851e+06,7.292602e+06,231611,41,634.550685,0.064613,229679.303001,14840.187942
...,...,...,...,...,...,...,...,...,...,...
83,2018,Tanker,4.297434e+06,6.534914e+06,121736,10,333.523288,0.029983,429743.384872,12884.958885
84,2018,Tractor,3.674957e+04,4.430740e+06,166834,2,457.079452,0.004376,18374.782570,80.400825
85,2018,Trade Plates,0.000000e+00,1.267065e+02,40,0,0.109589,0.000000,0.000000,0.000000
86,2018,Trailers And Semitrailers,4.280320e+07,1.405302e+07,742595,109,2034.506849,0.053576,392689.885206,21038.610660


In [17]:
risk_prev['pricing_year']=risk_prev['begin_year']+1

In [18]:
risk_prev

,begin_year,TYPE_VEHICLE,sum_claims_adj,sum_premium_adj,sum_Policy_duration,count_claims,exposure,frequency,severity,expected_loss,pricing_year
0,2011,Automobile,1.302647e+08,5.571959e+07,4769338,861,13066.679452,0.065893,151294.678552,9969.228886,2012
1,2011,Bus,5.833444e+08,1.190773e+08,3189598,954,8738.624658,0.109170,611472.153401,66754.719101,2012
2,2011,Motor-Cycle,1.183300e+06,8.457813e+06,1957333,73,5362.556164,0.013613,16209.583027,220.659612,2012
3,2011,Pick-Up,2.102559e+08,7.897389e+07,3939460,1060,10793.041096,0.098211,198354.635264,19480.692375,2012
4,2011,Special Construction,9.416851e+06,7.292602e+06,231611,41,634.550685,0.064613,229679.303001,14840.187942,2012
...,...,...,...,...,...,...,...,...,...,...,...
83,2018,Tanker,4.297434e+06,6.534914e+06,121736,10,333.523288,0.029983,429743.384872,12884.958885,2019
84,2018,Tractor,3.674957e+04,4.430740e+06,166834,2,457.079452,0.004376,18374.782570,80.400825,2019
85,2018,Trade Plates,0.000000e+00,1.267065e+02,40,0,0.109589,0.000000,0.000000,0.000000,2019
86,2018,Trailers And Semitrailers,4.280320e+07,1.405302e+07,742595,109,2034.506849,0.053576,392689.885206,21038.610660,2019


**now risk calculation with all previous years**

In [19]:
risk_all_prev = risk_by_year_vehicle_raw.sort_values(
    ["TYPE_VEHICLE", "begin_year"]
).copy()

risk_all_prev["cum_claims_prev"] = ( risk_all_prev.groupby("TYPE_VEHICLE")["sum_claims_adj"].cumsum()
    - risk_all_prev["sum_claims_adj"] # we only wants previous years and not the actual one
)

risk_all_prev["cum_exposure_prev"] = ( risk_all_prev.groupby("TYPE_VEHICLE")["exposure"].cumsum()
    - risk_all_prev["exposure"]
)

risk_all_prev["cum_count_claims_prev"] = ( risk_all_prev.groupby("TYPE_VEHICLE")["count_claims"].cumsum()
    - risk_all_prev["count_claims"]
)

risk_all_prev["all_prev_expected_loss"] = np.where(
    risk_all_prev["cum_exposure_prev"] > 0,
    risk_all_prev["cum_claims_prev"] / risk_all_prev["cum_exposure_prev"],
    np.nan
)

In [20]:
risk_all_prev["frequency_all_prev"] = risk_all_prev["cum_count_claims_prev"]/ risk_all_prev["cum_exposure_prev"]

risk_all_prev["severity_all_prev"] = np.where(
    risk_all_prev["cum_count_claims_prev"] > 0,
    risk_all_prev["cum_claims_prev"] / risk_all_prev["cum_count_claims_prev"],
    np.nan
)

risk_all_prev["expected_loss_all_prev"] = (risk_all_prev["frequency_all_prev"] * risk_all_prev["severity_all_prev"] )

In [21]:
risk_all_prev

,begin_year,TYPE_VEHICLE,sum_claims_adj,sum_premium_adj,sum_Policy_duration,count_claims,exposure,cum_claims_prev,cum_exposure_prev,cum_count_claims_prev,all_prev_expected_loss,frequency_all_prev,severity_all_prev,expected_loss_all_prev
0,2011,Automobile,1.302647e+08,5.571959e+07,4769338,861,13066.679452,0.000000e+00,0.000000,0,NaN,NaN,NaN,NaN
11,2012,Automobile,1.403965e+08,6.815366e+07,5772950,1260,15816.301370,1.302647e+08,13066.679452,861,9969.228886,0.065893,151294.678552,9969.228886
22,2013,Automobile,1.615153e+08,6.330945e+07,5224797,1184,14314.512329,2.706613e+08,28882.980822,2121,9370.959778,0.073434,127610.208182,9370.959778
33,2014,Automobile,1.689448e+08,6.645016e+07,6164047,1400,16887.800000,4.321766e+08,43197.493151,3305,10004.668365,0.076509,130764.475998,10004.668365
44,2015,Automobile,1.507637e+08,6.715270e+07,6269526,1434,17176.783562,6.011214e+08,60085.293151,4705,10004.468573,0.078305,127762.258662,10004.468573
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43,2014,Truck,1.587389e+09,3.615143e+08,6761952,1833,18525.895890,3.066034e+09,43990.413699,3914,69697.786009,0.088974,783350.649059,69697.786009
54,2015,Truck,1.634525e+09,3.933457e+08,7701085,2128,21098.863014,4.653424e+09,62516.309589,5747,74435.355835,0.091928,809713.546152,74435.355835
65,2016,Truck,1.433374e+09,4.146451e+08,8395611,2283,23001.673973,6.287949e+09,83615.172603,7875,75201.054292,0.094181,798469.731364,75201.054292
76,2017,Truck,9.024049e+08,4.026254e+08,9198171,2066,25200.468493,7.721324e+09,106616.846575,10158,72421.233572,0.095276,760122.420605,72421.233572


**start merging**

In [22]:
performance_df = performance_df.merge(
    risk_prev[
        ["pricing_year", "TYPE_VEHICLE", "expected_loss"]
    ],
    left_on=["begin_year", "TYPE_VEHICLE"],
    right_on=["pricing_year", "TYPE_VEHICLE"],
    how="left"
)

In [23]:
performance_df

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration,factor_to_2017_base,Claim_Adj,Premium_Adj,Insured_Adj,exposure,pricing_year,expected_loss
0,2013-08-08,519755.22,7209.140,Pick-Up,0.00,0,2013,364,1.382345,0.000000,9965.520684,718481.177441,0.997260,2013.0,21444.029282
1,2012-08-08,519755.22,7203.890,Pick-Up,0.00,0,2012,364,1.485468,0.000000,10701.149819,772079.873278,0.997260,2012.0,19480.692375
2,2011-08-08,519755.22,7045.804,Pick-Up,0.00,0,2011,365,1.836039,0.000000,12936.369154,954290.723372,1.000000,NaN,NaN
3,2011-07-08,519755.22,287.250,Pick-Up,0.00,0,2011,30,1.836039,0.000000,527.402130,954290.723372,0.082192,NaN,NaN
4,2013-08-08,285451.24,4286.900,Pick-Up,19894.43,1,2013,364,1.382345,27500.971498,5925.976000,394592.175558,0.997260,2013.0,21444.029282
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
801643,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,2018.0,11953.526061
801644,2018-02-02,0.00,299.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,263.120443,0.000000,0.997260,2018.0,11953.526061
801645,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,2018.0,11953.526061
801646,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,2018.0,11953.526061


In [24]:
performance_df=performance_df.drop(columns='pricing_year')

In [25]:
performance_df['Premium_Adj'].sum()

np.float64(7249189458.12035)

In [26]:
performance_df['exposure'].sum()

np.float64(771877.8219178084)

In [27]:
performance_df['Claim_Adj'].sum()

np.float64(19514429726.142464)

In [28]:
loss_ratio=performance_df['Claim_Adj'].sum()/performance_df['Premium_Adj'].sum()
loss_ratio

np.float64(2.691946436064368)

**now calculating the simulated premium using previous year**

In [29]:
performance_df['Sim_Premium_prev'] = np.where(
    performance_df['expected_loss'].isna() | ( (performance_df['expected_loss']* performance_df['exposure']) <= performance_df['Premium_Adj']),
    performance_df['Premium_Adj'],
    ( performance_df['expected_loss'] * performance_df['exposure'])
)

In [30]:
performance_df

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration,factor_to_2017_base,Claim_Adj,Premium_Adj,Insured_Adj,exposure,expected_loss,Sim_Premium_prev
0,2013-08-08,519755.22,7209.140,Pick-Up,0.00,0,2013,364,1.382345,0.000000,9965.520684,718481.177441,0.997260,21444.029282,21385.278517
1,2012-08-08,519755.22,7203.890,Pick-Up,0.00,0,2012,364,1.485468,0.000000,10701.149819,772079.873278,0.997260,19480.692375,19427.320615
2,2011-08-08,519755.22,7045.804,Pick-Up,0.00,0,2011,365,1.836039,0.000000,12936.369154,954290.723372,1.000000,NaN,12936.369154
3,2011-07-08,519755.22,287.250,Pick-Up,0.00,0,2011,30,1.836039,0.000000,527.402130,954290.723372,0.082192,NaN,527.402130
4,2013-08-08,285451.24,4286.900,Pick-Up,19894.43,1,2013,364,1.382345,27500.971498,5925.976000,394592.175558,0.997260,21444.029282,21385.278517
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
801643,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,11953.526061,11920.776675
801644,2018-02-02,0.00,299.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,263.120443,0.000000,0.997260,11953.526061,11920.776675
801645,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,11953.526061,11920.776675
801646,2018-02-02,0.00,294.510,Pick-Up,0.00,0,2018,364,0.878503,0.000000,258.727928,0.000000,0.997260,11953.526061,11920.776675


In [31]:
sim_loss_ratio_prev=performance_df['Claim_Adj'].sum()/performance_df['Sim_Premium_prev'].sum()
sim_loss_ratio_prev

np.float64(0.9158063495733633)

In [32]:
risk_all_prev.head()

,begin_year,TYPE_VEHICLE,sum_claims_adj,sum_premium_adj,sum_Policy_duration,count_claims,exposure,cum_claims_prev,cum_exposure_prev,cum_count_claims_prev,all_prev_expected_loss,frequency_all_prev,severity_all_prev,expected_loss_all_prev
0,2011,Automobile,1.302647e+08,5.571959e+07,4769338,861,13066.679452,0.000000e+00,0.000000,0,NaN,NaN,NaN,NaN
11,2012,Automobile,1.403965e+08,6.815366e+07,5772950,1260,15816.301370,1.302647e+08,13066.679452,861,9969.228886,0.065893,151294.678552,9969.228886
22,2013,Automobile,1.615153e+08,6.330945e+07,5224797,1184,14314.512329,2.706613e+08,28882.980822,2121,9370.959778,0.073434,127610.208182,9370.959778
33,2014,Automobile,1.689448e+08,6.645016e+07,6164047,1400,16887.800000,4.321766e+08,43197.493151,3305,10004.668365,0.076509,130764.475998,10004.668365
44,2015,Automobile,1.507637e+08,6.715270e+07,6269526,1434,17176.783562,6.011214e+08,60085.293151,4705,10004.468573,0.078305,127762.258662,10004.468573


In [33]:
performance_df = performance_df.merge(
    risk_all_prev[
        ["begin_year", "TYPE_VEHICLE", "expected_loss_all_prev"]
    ],
    on=["begin_year", "TYPE_VEHICLE"],
    how="left"
)

In [34]:
performance_df.head()

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration,factor_to_2017_base,Claim_Adj,Premium_Adj,Insured_Adj,exposure,expected_loss,Sim_Premium_prev,expected_loss_all_prev
0,2013-08-08,519755.22,7209.140,Pick-Up,0.00,0,2013,364,1.382345,0.000000,9965.520684,718481.177441,0.997260,21444.029282,21385.278517,20677.776517
1,2012-08-08,519755.22,7203.890,Pick-Up,0.00,0,2012,364,1.485468,0.000000,10701.149819,772079.873278,0.997260,19480.692375,19427.320615,19480.692375
2,2011-08-08,519755.22,7045.804,Pick-Up,0.00,0,2011,365,1.836039,0.000000,12936.369154,954290.723372,1.000000,NaN,12936.369154,NaN
3,2011-07-08,519755.22,287.250,Pick-Up,0.00,0,2011,30,1.836039,0.000000,527.402130,954290.723372,0.082192,NaN,527.402130,NaN
4,2013-08-08,285451.24,4286.900,Pick-Up,19894.43,1,2013,364,1.382345,27500.971498,5925.976000,394592.175558,0.997260,21444.029282,21385.278517,20677.776517


In [35]:
performance_df['Sim_Premium_all_prev'] = np.where(
    performance_df['expected_loss_all_prev'].isna() | ( (performance_df['expected_loss_all_prev']* performance_df['exposure']) <= performance_df['Premium_Adj']),
    performance_df['Premium_Adj'],
    ( performance_df['expected_loss_all_prev'] * performance_df['exposure'])
)

In [36]:
performance_df.head()

,INSR_BEGIN,INSURED_VALUE,PREMIUM,TYPE_VEHICLE,CLAIM_PAID,has_claim,begin_year,policy_duration,factor_to_2017_base,Claim_Adj,Premium_Adj,Insured_Adj,exposure,expected_loss,Sim_Premium_prev,expected_loss_all_prev,Sim_Premium_all_prev
0,2013-08-08,519755.22,7209.140,Pick-Up,0.00,0,2013,364,1.382345,0.000000,9965.520684,718481.177441,0.997260,21444.029282,21385.278517,20677.776517,20621.125075
1,2012-08-08,519755.22,7203.890,Pick-Up,0.00,0,2012,364,1.485468,0.000000,10701.149819,772079.873278,0.997260,19480.692375,19427.320615,19480.692375,19427.320615
2,2011-08-08,519755.22,7045.804,Pick-Up,0.00,0,2011,365,1.836039,0.000000,12936.369154,954290.723372,1.000000,NaN,12936.369154,NaN,12936.369154
3,2011-07-08,519755.22,287.250,Pick-Up,0.00,0,2011,30,1.836039,0.000000,527.402130,954290.723372,0.082192,NaN,527.402130,NaN,527.402130
4,2013-08-08,285451.24,4286.900,Pick-Up,19894.43,1,2013,364,1.382345,27500.971498,5925.976000,394592.175558,0.997260,21444.029282,21385.278517,20677.776517,20621.125075


In [37]:
simulation_summary = (
    performance_df.groupby("TYPE_VEHICLE")
    .agg(
        claims_adj=("Claim_Adj", "sum"),
        premium_adj=("Premium_Adj", "sum"),
        exposure=("exposure", "sum"),
        simulated_premium_prev=("Sim_Premium_prev", "sum"),
        simulated_premium_all_prev=("Sim_Premium_all_prev", "sum")
    )
    .reset_index()
)
simulation_summary

,TYPE_VEHICLE,claims_adj,premium_adj,exposure,simulated_premium_prev,simulated_premium_all_prev
0,Automobile,1.042645e+09,4.824519e+08,122373.564384,1.133367e+09,1.176477e+09
1,Bus,3.647939e+09,1.385173e+09,101551.668493,4.113002e+09,4.831701e+09
2,Motor-Cycle,3.299866e+07,1.025208e+08,141944.254795,1.064263e+08,1.041214e+08
3,Pick-Up,2.587492e+09,1.034275e+09,138519.980822,2.781219e+09,2.852479e+09
4,Special Construction,1.964590e+08,1.562769e+08,11741.775342,2.744769e+08,2.580479e+08
5,Station Wagones,1.321889e+09,8.416877e+08,58503.761644,1.540962e+09,1.585305e+09
6,Tanker,5.632423e+08,2.241089e+08,9972.594521,6.406835e+08,7.301075e+08
7,Tractor,3.378503e+07,1.008256e+08,11152.282192,1.090884e+08,1.094475e+08
8,Trade Plates,0.000000e+00,2.194317e+04,17.076712,2.194317e+04,2.194317e+04
9,Trailers And Semitrailers,1.307013e+09,2.928054e+08,34683.501370,1.275615e+09,1.174439e+09


In [38]:
simulated_pricing=performance_df.copy()

In [39]:
simulated_pricing.to_csv('simulated_pricing.csv', index=False)